# 放心借 lookalike：导出训练数据 + 训练

## 怎么在云分析机里用这个 Notebook

1. 打开 Jupyter / 云分析机「Notebook」页面，点 **新建 → Python 3**。
2. 菜单 **File → Save As**，名字例如 `fxj_lookalike_train.ipynb`。
3. 从本仓库复制下面 **每一个代码格**（共 5 个），按顺序 **粘贴到新建 Notebook 的单元格里**（一格对应一个 Cell）。
4. 先改 **第 2 格**里的 `TABLE_NAME`、输出路径等参数。
5. 从上到下点 **运行**（或 Shift+Enter），**一次只跑一格**，等上一格跑完再跑下一格。

**不要**在 Notebook 里用 `%run export_training_data_cloud.py`，会报 `unrecognized arguments: -f ...`。

也可以：把仓库里的 `model/notebooks/export_training_data.ipynb` 整个上传到 Jupyter，直接打开运行。

---
## 第 1 格：安装依赖（首次运行一次即可）

In [ ]:
!pip install -q lightgbm pulearn pyarrow pyyaml scikit-learn joblib pandas numpy

---
## 第 2 格：参数（必改表名）

In [ ]:
import os
import sys
from pathlib import Path

import dtools

# ===== 按你们环境修改 =====
TABLE_NAME = "ai_decision_dev.fxj_lookalike_pu_training"
# TABLE_NAME = "lj_iceberg.ai_decision_dev.fxj_lookalike_pu_training"

# 工作目录：建议 cd 到上传的 model 目录；或写绝对路径
MODEL_ROOT = Path(".").resolve()  # 若 notebook 在 model 目录下
# MODEL_ROOT = Path("/home/finance/你的路径/放心借客群lookalike/model")

OUTPUT_PARQUET = MODEL_ROOT / "data" / "training_pu.parquet"

# 试跑：设 LIMIT_ROWS = 50000；正式全量设 None
LIMIT_ROWS = None
# 背景太多可先抽 20% 未标注行（种子全保留）
UNLABELED_SAMPLE_FRAC = 1.0
MS13_MIN_FOR_UNLABELED = 0

sys.path.insert(0, str(MODEL_ROOT / "src"))
print("MODEL_ROOT:", MODEL_ROOT)
print("输出:", OUTPUT_PARQUET)

---
## 第 3 格：从 Hive 拉数并保存 parquet

In [ ]:
extra = ""
if MS13_MIN_FOR_UNLABELED and MS13_MIN_FOR_UNLABELED > 0:
    extra += f"\n  AND (pu_label = 1 OR ms13_score >= {float(MS13_MIN_FOR_UNLABELED)})"
if UNLABELED_SAMPLE_FRAC < 1.0:
    extra += f"\n  AND (pu_label = 1 OR rand() < {float(UNLABELED_SAMPLE_FRAC)})"
limit_sql = f"\nLIMIT {int(LIMIT_ROWS)}" if LIMIT_ROWS else ""

query = f"""
SELECT *
FROM {TABLE_NAME}
WHERE dataset_split IN ('train', 'val')
{extra}
{limit_sql}
""".strip()

print(query)
df_data = dtools.get_as_frame(query)
print(f"行数: {len(df_data)}, 列数: {len(df_data.columns)}")
print(df_data["pu_label"].value_counts())

OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_data.to_parquet(OUTPUT_PARQUET, index=False)
print("已保存:", OUTPUT_PARQUET)

---
## 第 4 格：训练（需已上传 model/src 与 config.yaml）

In [ ]:
import json

from config_loader import load_config, resolve_path
from dataset import (
    apply_filters,
    feature_group_summary,
    load_table,
    prepare_splits,
    subsample_unlabeled,
    to_xy,
)
from train_pu import (
    save_sklearn_pu_artifacts,
    save_lgbm_artifacts,
    train_elkanoto_pu,
    train_weighted_naive_pu,
)

config_path = MODEL_ROOT / "config.yaml"
cfg = load_config(config_path)
data_path = resolve_path(cfg["data"]["input_path"], config_path)
if not data_path.exists():
    data_path = OUTPUT_PARQUET

print("加载:", data_path)
df = load_table(data_path)
df = apply_filters(df, cfg)
label_col = cfg["data"]["label_col"]
print(f"过滤后 {len(df)}  P={int((df[label_col]==1).sum())}  U={int((df[label_col]==0).sum())}")

train_df, val_df, feature_columns = prepare_splits(df, cfg)
ratio = float(cfg["training"].get("unlabeled_subsample_ratio", 1.0))
seed = int(cfg["training"]["random_seed"])
train_df = subsample_unlabeled(train_df, label_col, ratio, seed)
print(f"train={len(train_df)} val={len(val_df)} features={len(feature_columns)}")

x_train, y_train = to_xy(train_df, feature_columns, label_col)
x_val, y_val = to_xy(val_df, feature_columns, label_col)

pu_cfg = cfg["pu"]
train_cfg = cfg["training"]
method = pu_cfg.get("method", "elkanoto")
out_dir = resolve_path(cfg["output"]["artifacts_dir"], config_path)

if method == "elkanoto":
    clf, metrics = train_elkanoto_pu(
        x_train,
        y_train,
        x_val,
        y_val,
        params=train_cfg["params"],
        hold_out_ratio=float(pu_cfg.get("hold_out_ratio", 0.1)),
        seed=seed,
    )
    manifest = save_sklearn_pu_artifacts(
        clf, metrics, feature_columns, out_dir, cfg["output"]["model_name"]
    )
else:
    booster, metrics = train_weighted_naive_pu(
        x_train,
        y_train,
        x_val,
        y_val,
        params=dict(train_cfg["params"]),
        unlabeled_weight=float(pu_cfg.get("unlabeled_weight", 0.05)),
        num_boost_round=int(train_cfg["num_boost_round"]),
        early_stopping_rounds=int(train_cfg["early_stopping_rounds"]),
        seed=seed,
    )
    manifest = save_lgbm_artifacts(
        booster, metrics, feature_columns, out_dir, cfg["output"]["model_name"]
    )

print(json.dumps(metrics["val"], ensure_ascii=False, indent=2))
print("模型:", manifest["model_path"])